In [1]:
!pip install -q PyMuPDF sentence-transformers faiss-cpu tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.1 MB/s eta 0:00:00


In [2]:
import os, re, json, time, random, math
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
SEED = 42
PDF_PATH = "human-nutrition-text.pdf"
PDF_URL = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"
PAGE_OFFSET = 41                 # the PDF's printed page 1 is PDF index 41
EMBED_MODEL = "all-mpnet-base-v2"
QGEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # ungated, no Hugging Face token needed
N_EVAL_QUESTIONS = 60
USE_MANUAL_QUESTIONS = False
MANUAL_QUESTIONS = [             # (question, PRINTED page number where the answer is)
    # ("What are micronutrients?", 123),
]
FAISS_STRATEGY = "sentence_groups_10"       # which chunk set to index for the FAISS test
RUN_SCALE_TEST = True
SCALE_N = 100_000
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "NO GPU - switch runtime to GPU")

device: cuda | Tesla T4


In [3]:
import requests, fitz

if not os.path.exists(PDF_PATH):
    r = requests.get(PDF_URL, timeout=300)
    r.raise_for_status()
    open(PDF_PATH, "wb").write(r.content)

doc = fitz.open(PDF_PATH)
pages = []
for i, p in enumerate(tqdm(doc, desc="reading pages")):
    t = re.sub(r"\s+", " ", p.get_text().replace("\n", " ")).strip()
    pages.append({"idx": i, "printed": i - PAGE_OFFSET, "text": t})
print(len(pages), "pages | avg words/page:", int(np.mean([len(p["text"].split()) for p in pages])))

reading pages:   0%|          | 0/1208 [00:00<?, ?it/s]

1208 pages | avg words/page: 171


In [4]:
def split_sentences(text):
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"\u201c\u2018\'(])', text)
    return [s.strip() for s in parts if len(s.strip()) > 3]

def chunk_sentence_groups(pages, size=10, min_tokens=30):
    out = []
    for pg in pages:
        sents = split_sentences(pg["text"])
        for i in range(0, len(sents), size):
            txt = " ".join(sents[i:i + size]).strip()
            if len(txt) / 4 > min_tokens:
                out.append({"text": txt, "p0": pg["idx"], "p1": pg["idx"]})
    return out

def chunk_chapters(pages):
    starts = [i for i, p in enumerate(pages) if re.search(r"university\s+of\s+hawai", p["text"], re.I)]
    if not starts:
        return [{"text": " ".join(p["text"] for p in pages), "p0": 0, "p1": len(pages) - 1}]
    if starts[0] > 0:
        starts = [0] + starts
    out = []
    for k, s in enumerate(starts):
        e = starts[k + 1] - 1 if k + 1 < len(starts) else len(pages) - 1
        if e < s:
            continue
        txt = " ".join(p["text"] for p in pages[s:e + 1]).strip()
        if txt:
            out.append({"text": txt, "p0": pages[s]["idx"], "p1": pages[e]["idx"]})
    return out

def chunk_fixed_windows(pages, words_per=200, overlap=40):
    words, wpage = [], []
    for pg in pages:
        ws = pg["text"].split()
        words += ws
        wpage += [pg["idx"]] * len(ws)
    out, step = [], words_per - overlap
    for s in range(0, len(words), step):
        w = words[s:s + words_per]
        if len(w) < 30:
            break
        out.append({"text": " ".join(w), "p0": wpage[s], "p1": wpage[s + len(w) - 1]})
    return out

def chunk_semantic(pages, encode_fn, percentile=25, max_sents=12, min_tokens=30):
    recs = [(pg["idx"], s) for pg in pages for s in split_sentences(pg["text"])]
    sents = [s for _, s in recs]
    emb = encode_fn(sents)
    sims = (emb[:-1] * emb[1:]).sum(1)
    thr = np.percentile(sims, percentile)
    out, start = [], 0
    for i in range(len(sents)):
        if i == len(sents) - 1 or sims[i] < thr or (i - start + 1) >= max_sents:
            txt = " ".join(sents[start:i + 1])
            if len(txt) / 4 > min_tokens:
                out.append({"text": txt, "p0": recs[start][0], "p1": recs[i][0]})
            start = i + 1
    return out

def evaluate_retrieval(q_emb, E, chunks, gold, tok_counts, ks=(1, 3, 5), focus_span=3):
    scores = q_emb @ E.T
    order = np.argsort(-scores, axis=1)[:, :max(ks)]
    n = len(gold)
    first = np.full(n, np.inf)
    first_focus = np.full(n, np.inf)
    ctx = []
    for qi in range(n):
        for r, ci in enumerate(order[qi]):
            c = chunks[ci]
            if c["p0"] <= gold[qi] <= c["p1"]:
                if first[qi] == np.inf:
                    first[qi] = r + 1
                if (c["p1"] - c["p0"] + 1) <= focus_span and first_focus[qi] == np.inf:
                    first_focus[qi] = r + 1
        ctx.append(sum(tok_counts[ci] for ci in order[qi][:5]))
    res = {f"hit@{k}": float(np.mean(first <= k)) for k in ks}
    res["mrr@5"] = float(np.mean(np.where(first <= 5, 1.0 / first, 0.0)))
    res["focused_hit@5"] = float(np.mean(first_focus <= 5))
    res["avg_tokens_top5"] = float(np.mean(ctx))
    return res, first

def bootstrap_ci(first, k=5, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    hits = (first <= k).astype(float)
    means = [hits[rng.integers(0, len(hits), len(hits))].mean() for _ in range(n_boot)]
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))
import faiss
_KEEP_ALIVE = []

def build_flat(E):
    idx = faiss.IndexFlatIP(E.shape[1]); idx.add(E); return idx

def build_ivf(E, nlist):
    quant = faiss.IndexFlatIP(E.shape[1])
    idx = faiss.IndexIVFFlat(quant, E.shape[1], nlist, faiss.METRIC_INNER_PRODUCT)
    idx.train(E); idx.add(E)
    _KEEP_ALIVE.append(quant)   # faiss needs the quantizer to stay alive
    return idx

def build_hnsw(E, M=32, ef_construction=200):
    idx = faiss.IndexHNSWFlat(E.shape[1], M, faiss.METRIC_INNER_PRODUCT)
    idx.hnsw.efConstruction = ef_construction
    idx.add(E)
    return idx

def timed_search(index, Q, k, repeats=5):
    index.search(Q, k)
    times = []
    for _ in range(repeats):
        t = time.perf_counter(); D, I = index.search(Q, k); times.append(time.perf_counter() - t)
    return I, float(np.median(times) / len(Q) * 1000)    # ms per query

def recall_vs_truth(I, truth, k):
    return float(np.mean([len(set(I[i][:k]) & set(truth[i][:k])) / k for i in range(len(I))]))

def ann_benchmark(E, Q, nlist, ks=(5, 10), ivf_probes=(1, 2, 4, 8, 16), hnsw_efs=(8, 16, 32, 64, 128),
                  M=32, ef_construction=200):
    faiss.omp_set_num_threads(1)   # fair per-query latency
    kmax = max(ks)
    flat = build_flat(E)
    truth, flat_ms = timed_search(flat, Q, kmax)
    rows = [{"index": "Flat (exact)", "setting": "-", **{f"recall@{k}": 1.0 for k in ks}, "ms_per_query": flat_ms}]
    t0 = time.perf_counter(); ivf = build_ivf(E, nlist); ivf_build = time.perf_counter() - t0
    for p in ivf_probes:
        if p > nlist: continue
        ivf.nprobe = p
        I, ms = timed_search(ivf, Q, kmax)
        rows.append({"index": f"IVFFlat (nlist={nlist})", "setting": f"nprobe={p}",
                     **{f"recall@{k}": recall_vs_truth(I, truth, k) for k in ks}, "ms_per_query": ms})
    t0 = time.perf_counter(); hnsw = build_hnsw(E, M, ef_construction); hnsw_build = time.perf_counter() - t0
    for ef in hnsw_efs:
        hnsw.hnsw.efSearch = ef
        I, ms = timed_search(hnsw, Q, kmax)
        rows.append({"index": f"HNSW (M={M})", "setting": f"efSearch={ef}",
                     **{f"recall@{k}": recall_vs_truth(I, truth, k) for k in ks}, "ms_per_query": ms})
    faiss.omp_set_num_threads(os.cpu_count() or 1)
    return pd.DataFrame(rows), {"ivf_build_s": ivf_build, "hnsw_build_s": hnsw_build}
print("helpers ready")

helpers ready


In [6]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(EMBED_MODEL, device=device)
print("embedding model max tokens:", model.max_seq_length)

def encode(texts, bs=64, show=False):
    return model.encode(texts, batch_size=bs, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=show).astype("float32")

def count_tokens(texts):
    return [len(x) for x in model.tokenizer(texts, add_special_tokens=False, truncation=False)["input_ids"]]

chunk_sets = {
    "sentence_groups_10":   chunk_sentence_groups(pages),
    "chapter_structural":   chunk_chapters(pages),
    "fixed_200w_overlap40": chunk_fixed_windows(pages),
}
print("building semantic chunks")
chunk_sets["semantic_breakpoint"] = chunk_semantic(pages, lambda s: encode(s, bs=128, show=True))

embs, toks, stat_rows = {}, {}, []
for name, ch in chunk_sets.items():
    texts = [c["text"] for c in ch]
    t0 = time.perf_counter(); embs[name] = encode(texts); et = time.perf_counter() - t0
    toks[name] = count_tokens(texts)
    spans = [c["p1"] - c["p0"] + 1 for c in ch]
    stat_rows.append({"strategy": name, "n_chunks": len(ch), "mean_tokens": round(float(np.mean(toks[name])), 1),
                      "median_tokens": float(np.median(toks[name])),
                      "pct_over_384_tokens": round(100 * float(np.mean(np.array(toks[name]) > 384)), 1),
                      "mean_pages_spanned": round(float(np.mean(spans)), 2), "embed_seconds": round(et, 1)})
chunk_stats = pd.DataFrame(stat_rows)
print(chunk_stats.to_string(index=False))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding model max tokens: 384
building semantic chunks


Batches:   0%|          | 0/93 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (406 > 384). Running this sequence through the model will result in indexing errors


            strategy  n_chunks  mean_tokens  median_tokens  pct_over_384_tokens  mean_pages_spanned  embed_seconds
  sentence_groups_10      1605        184.3          189.0                  0.8                1.00           26.8
  chapter_structural       172       1741.3         1185.0                 76.7                7.02            4.9
fixed_200w_overlap40      1299        287.6          280.0                  2.7                2.15           36.5
 semantic_breakpoint      1651        170.2          140.0                  4.1                1.50           28.5


In [7]:
def usable_page(p):
    t = p["text"]
    if not (800 <= len(t) <= 4000): return False
    if sum(c.isalpha() for c in t) / len(t) < 0.75: return False
    if len(re.findall(r"Accessed|http|www\.", t)) > 2: return False
    return True
eval_set = []
if USE_MANUAL_QUESTIONS:
    assert MANUAL_QUESTIONS, "Fill MANUAL_QUESTIONS or set USE_MANUAL_QUESTIONS = False"
    for q, printed in MANUAL_QUESTIONS:
        eval_set.append({"question": q, "gold_idx": printed + PAGE_OFFSET, "source": "manual"})
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    qtok = AutoTokenizer.from_pretrained(QGEN_MODEL)
    qlm = AutoModelForCausalLM.from_pretrained(
        QGEN_MODEL, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device)

    def gen_question(passage):
        msgs = [{"role": "system", "content": "You write exam questions for a nutrition textbook."},
                {"role": "user", "content": "Read the passage and write ONE specific question that can be answered "
                 "using only this passage. Do not mention 'the passage', 'the text' or 'the author'. Paraphrase; "
                 "do not copy long phrases. Output only the question.\n\nPassage:\n" + passage[:1800]}]
        prompt = qtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = qtok(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            out = qlm.generate(**inp, max_new_tokens=60, do_sample=False)
        q = qtok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        return q.split("\n")[0].strip()

    cands = [p for p in pages if usable_page(p)]
    random.Random(SEED).shuffle(cands)
    for p in tqdm(cands, desc="generating questions"):
        q = gen_question(p["text"])
        if q.endswith("?") and 5 <= len(q.split()) <= 40:
            eval_set.append({"question": q, "gold_idx": p["idx"], "source": "synthetic"})
        if len(eval_set) >= N_EVAL_QUESTIONS:
            break
    del qlm; torch.cuda.empty_cache()

eval_df = pd.DataFrame(eval_set)
eval_df["printed_page"] = eval_df["gold_idx"] - PAGE_OFFSET
eval_df.to_csv("eval_questions.csv", index=False)
print(len(eval_df), "questions saved to eval_questions.csv\n")
for _, r in eval_df.head(12).iterrows():
    print(f"p.{r.printed_page}: {r.question}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

generating questions:   0%|          | 0/579 [00:00<?, ?it/s]

60 questions saved to eval_questions.csv

p.223: What is the primary reason for replacing lost sodium during prolonged intense exercise?
p.741: What is the main difference between the Food Pyramid and MyPlate programs regarding their approach to promoting optimal health through dietary recommendations?
p.73: How does the closure of the epiglottis during swallowing prevent choking?
p.948: What type of physical activity motivates individuals most effectively according to the passage?
p.92: What is the relationship between electrolyte levels such as calcium, chloride, magnesium, and potassium with various health conditions mentioned in the passage?
p.626: What are some plant-based foods that are high in calcium according to the passage?
p.922: What is the recommended daily intake of vitamin B12 for older adults compared to younger adults?
p.465: What is the primary difference between the pathways for breaking down glucose, fatty acids, and amino acids?
p.793: What additional nutrient is c

In [8]:
q_emb = encode(eval_df["question"].tolist())
gold = eval_df["gold_idx"].to_numpy()

rows = []
for name, ch in chunk_sets.items():
    res, first = evaluate_retrieval(q_emb, embs[name], ch, gold, toks[name])
    lo, hi = bootstrap_ci(first, k=5)
    rows.append({"strategy": name, **{k: round(v, 3) for k, v in res.items()},
                 "hit@5_95%CI": f"[{lo:.2f}, {hi:.2f}]"})
retrieval_df = pd.DataFrame(rows)
retrieval_df.to_csv("retrieval_results.csv", index=False)
print(retrieval_df.to_string(index=False))
print("""
How to read this:
- hit@k: the answer page falls inside one of the top-k retrieved chunks.
- focused_hit@5: same, but ignoring chunks that span more than 3 pages (so huge chunks cannot win by size).
- avg_tokens_top5: how much text you would feed the LLM. Big = slower and costlier generation.
- With ~60 questions the confidence interval is wide; differences under ~0.08 are probably noise.""")

            strategy  hit@1  hit@3  hit@5  mrr@5  focused_hit@5  avg_tokens_top5  hit@5_95%CI
  sentence_groups_10  0.550  0.767  0.817  0.662          0.817         1036.783 [0.72, 0.92]
  chapter_structural  0.350  0.567  0.650  0.466          0.100        10269.467 [0.52, 0.77]
fixed_200w_overlap40  0.717  0.767  0.800  0.749          0.783         1423.450 [0.68, 0.90]
 semantic_breakpoint  0.617  0.783  0.833  0.699          0.833         1092.183 [0.73, 0.92]

How to read this:
- hit@k: the answer page falls inside one of the top-k retrieved chunks.
- focused_hit@5: same, but ignoring chunks that span more than 3 pages (so huge chunks cannot win by size).
- avg_tokens_top5: how much text you would feed the LLM. Big = slower and costlier generation.
- With ~60 questions the confidence interval is wide; differences under ~0.08 are probably noise.


## FAISS: Flat vs IVFFlat vs HNSW

In [9]:
E = embs[FAISS_STRATEGY]
Q = np.tile(q_emb, (10, 1))  # tile queries so timings are stable
N = len(E)
nlist_small = max(4, int(math.sqrt(N)))
print(f"Index test on real embeddings: {N} vectors, dim {E.shape[1]}, nlist={nlist_small}, {len(Q)} queries")
small_df, small_build = ann_benchmark(E, Q, nlist=nlist_small)
small_df["ms_per_query"] = small_df["ms_per_query"].round(4)
small_df.to_csv("faiss_real.csv", index=False)
print(small_df.round(3).to_string(index=False))
print("build seconds:", {k: round(v, 3) for k, v in small_build.items()})

Index test on real embeddings: 1605 vectors, dim 768, nlist=40, 600 queries
             index      setting  recall@5  recall@10  ms_per_query
      Flat (exact)            -     1.000      1.000         0.134
IVFFlat (nlist=40)     nprobe=1     0.610      0.573         0.021
IVFFlat (nlist=40)     nprobe=2     0.797      0.758         0.060
IVFFlat (nlist=40)     nprobe=4     0.917      0.888         0.095
IVFFlat (nlist=40)     nprobe=8     0.967      0.960         0.160
IVFFlat (nlist=40)    nprobe=16     0.993      0.990         0.113
       HNSW (M=32)   efSearch=8     0.953      0.932         0.033
       HNSW (M=32)  efSearch=16     0.983      0.978         0.047
       HNSW (M=32)  efSearch=32     1.000      1.000         0.080
       HNSW (M=32)  efSearch=64     1.000      1.000         0.126
       HNSW (M=32) efSearch=128     1.000      1.000         0.220
build seconds: {'ivf_build_s': 0.123, 'hnsw_build_s': 0.495}


In [10]:
scale_df, scale_build = None, None
if RUN_SCALE_TEST:
    rng = np.random.default_rng(SEED)
    base = embs[FAISS_STRATEGY]
    big = base[rng.integers(0, len(base), SCALE_N)] + rng.normal(0, 0.015, (SCALE_N, base.shape[1]))
    big = (big / np.linalg.norm(big, axis=1, keepdims=True)).astype("float32")
    nlist_big = int(4 * math.sqrt(SCALE_N))
    print(f"SYNTHETIC scale test: {SCALE_N:,} vectors, nlist={nlist_big} (building HNSW takes a few minutes)")
    scale_df, scale_build = ann_benchmark(big, Q[:200], nlist=nlist_big, ivf_probes=(1, 4, 16, 32, 64),
                                          hnsw_efs=(16, 32, 64, 128), ef_construction=100)
    scale_df["ms_per_query"] = scale_df["ms_per_query"].round(4)
    scale_df.to_csv("faiss_scale_synthetic.csv", index=False)
    print(scale_df.round(3).to_string(index=False))
    print("build seconds:", {k: round(v, 1) for k, v in scale_build.items()})
    del big

SYNTHETIC scale test: 100,000 vectors, nlist=1264 (building HNSW takes a few minutes)
               index      setting  recall@5  recall@10  ms_per_query
        Flat (exact)            -     1.000      1.000         3.008
IVFFlat (nlist=1264)     nprobe=1     0.895      0.888         0.046
IVFFlat (nlist=1264)     nprobe=4     0.961      0.963         0.130
IVFFlat (nlist=1264)    nprobe=16     1.000      1.000         0.420
IVFFlat (nlist=1264)    nprobe=32     1.000      1.000         0.785
IVFFlat (nlist=1264)    nprobe=64     1.000      1.000         1.501
         HNSW (M=32)  efSearch=16     0.680      0.678         0.066
         HNSW (M=32)  efSearch=32     0.803      0.805         0.110
         HNSW (M=32)  efSearch=64     0.877      0.879         0.186
         HNSW (M=32) efSearch=128     0.970      0.970         0.312
build seconds: {'ivf_build_s': 23.4, 'hnsw_build_s': 29.9}


In [13]:
gpu = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
print(f"Environment: {gpu} | embedding model {EMBED_MODEL} | questions: {len(eval_df)} "
      f"({eval_df['source'].iloc[0]}) | seed {SEED}")
print("\n[CHUNK STATS]\n" + chunk_stats.to_string(index=False))
print("\n[RETRIEVAL BY CHUNKING STRATEGY]\n" + retrieval_df.to_string(index=False))
print(f"\n[FAISS ON REAL EMBEDDINGS: {N} vectors]\n" + small_df.round(3).to_string(index=False))
print("build seconds:", {k: round(v, 3) for k, v in small_build.items()})
if scale_df is not None:
    print(f"\n[FAISS ON SYNTHETIC SCALE-UP: {SCALE_N:,} vectors]\n" + scale_df.round(3).to_string(index=False))
    print("build seconds:", {k: round(v, 1) for k, v in scale_build.items()})
print("\n[SAMPLE QUESTIONS]")
for _, r in eval_df.head(8).iterrows():
    print(f"p.{r.printed_page}: {r.question}")
json.dump({"chunk_stats": chunk_stats.to_dict("records"), "retrieval": retrieval_df.to_dict("records"),
           "faiss_real": small_df.to_dict("records"),
           "faiss_scale": None if scale_df is None else scale_df.to_dict("records")},
          open("results_summary.json", "w"), indent=2)

Environment: Tesla T4 | embedding model all-mpnet-base-v2 | questions: 60 (synthetic) | seed 42

[CHUNK STATS]
            strategy  n_chunks  mean_tokens  median_tokens  pct_over_384_tokens  mean_pages_spanned  embed_seconds
  sentence_groups_10      1605        184.3          189.0                  0.8                1.00           26.8
  chapter_structural       172       1741.3         1185.0                 76.7                7.02            4.9
fixed_200w_overlap40      1299        287.6          280.0                  2.7                2.15           36.5
 semantic_breakpoint      1651        170.2          140.0                  4.1                1.50           28.5

[RETRIEVAL BY CHUNKING STRATEGY]
            strategy  hit@1  hit@3  hit@5  mrr@5  focused_hit@5  avg_tokens_top5  hit@5_95%CI
  sentence_groups_10  0.550  0.767  0.817  0.662          0.817         1036.783 [0.72, 0.92]
  chapter_structural  0.350  0.567  0.650  0.466          0.100        10269.467 [0.52, 0.77